# 04 - Qdrant Ingestion (Hue Foods RAG MVP)

Notebook này trình bày Phase 4: đưa 572 canonical food chunks cùng dense/sparse representations vào một active Qdrant collection có schema kiểm chứng được.

Các khái niệm chính:

- Named vectors: mỗi point có hai vector, `dense` (384 chiều, cosine) và `sparse` (TF-IDF, index được bật).
- Payload: `chunk_id`, `source`, `title`, `section`, `category`, `subcategory`, `chunk_type`, `embedding_model`, `embedding_dimension`.
- One-active-collection lifecycle: tại một thời điểm chỉ có một collection active cho Hue Foods; collection gắn chặt với embedding model và dimension.
- Point ID deterministic: `uuid5("hue-rag:<chunk_id>")` nên upsert idempotent.

Default mode của notebook này KHÔNG gọi Qdrant, KHÔNG load E5 và KHÔNG gọi network/model API. Real mode chỉ inspect read-only và phải opt-in bằng `HUE_RAG_QDRANT_REAL=1`. Notebook không chứa reset/delete cell.


In [ ]:
import sys
from pathlib import Path

for base in (Path.cwd(), *Path.cwd().parents):
    if (base / "backend").is_dir():
        sys.path.insert(0, str(base / "backend"))
        break
print(f"backend on path: {sys.path[0]}")


## Cấu hình vector database (`settings.yaml`)

Nhóm `vector_database` khai báo Qdrant local: URL, active collection `hue_foods_e5_small_384`, `reset_collection: false` (ingestion không bao giờ xóa), `vector_size: 384`, `distance: cosine`, timeout 30 giây, batch upsert 64 và tối đa một retry chỉ cho connection/timeout errors.

Expected output: `collection_name: hue_foods_e5_small_384`, `reset_collection: False`, `upsert_batch_size: 64`, `upsert_max_retries: 1`.


In [ ]:
from core.settings_loader import load_settings

settings = load_settings()
db = settings["vector_database"]
print("url:", db["url"])
print("collection_name:", db["collection_name"])
print("reset_collection:", db["reset_collection"])
print("vector_size:", db["vector_size"])
print("distance:", db["distance"])
print("timeout:", db["timeout"])
print("upsert_batch_size:", db["upsert_batch_size"])
print("upsert_max_retries:", db["upsert_max_retries"])


## Chunking và chunk_id (Phase 2)

`chunk_foods_markdown()` đọc curated foods Markdown và trả 572 chunk dicts. Mỗi chunk có `text` và `metadata` với `chunk_id` ổn định dạng `foods/<subcategory>/<file>.md|<section>|<index>`.

Expected output: `chunk count: 572` và 3 chunk_id mẫu đầu tiên.


In [ ]:
from ingestion.chunking.markdown_chunker import chunk_foods_markdown

chunks = chunk_foods_markdown()
print("chunk count:", len(chunks))
for chunk in chunks[:3]:
    print(chunk["metadata"]["chunk_id"])


## Point contract và UUID5 deterministic

Mỗi point ID là `uuid.uuid5(uuid.NAMESPACE_URL, f"hue-rag:{chunk_id}")`. Cùng một `chunk_id` luôn ra cùng một ID nên rerun sau partial failure là idempotent. Original `chunk_id` được giữ trong payload.

Expected output: hai lần gọi `point_id_for` cùng chunk_id cho cùng UUID; chunk_id khác cho UUID khác.


In [ ]:
from vectorstore.hybrid_index import point_id_for

chunk_id = "foods/restaurants/example.md|Tóm tắt|0"
first = point_id_for(chunk_id)
print("point id:", first)
print("deterministic:", point_id_for(chunk_id) == first)
print("different chunk:", point_id_for("foods/cafes/other.md|Tóm tắt|0") != first)


## Build sample point (safe default, offline)

`build_points` kết hợp chunks, dense vectors và sparse representations thành point dicts. Demo dưới đây dùng 2 chunk thật từ KB, vector dense giả deterministic (không load E5) và `SparseEmbedder` thật fit trên chính 2 chunk đó.

Expected output: 1 point mẫu có `id` là UUID5, vector names `dense`/`sparse`, dense dài 384 và payload có đủ 9 field chuẩn.


In [ ]:
import math

from embedding.sparse_embedder import SparseEmbedder
from vectorstore.hybrid_index import build_points

sample = chunks[:2]
texts = [chunk["text"] for chunk in sample]
sparse_embedder = SparseEmbedder().fit(texts)
dimension = settings["embedding"]["vector_size"]
norm = 1.0 / math.sqrt(dimension)
fake_dense = [[norm] * dimension for _ in sample]
sample_points = build_points(
    sample,
    fake_dense,
    [sparse_embedder.encode(text) for text in texts],
    settings["embedding"]["model"],
    dimension,
)
print("sample points:", len(sample_points))
point = sample_points[0]
print("point id:", point["id"])
print("vector names:", sorted(point["vector"]))
print("dense length:", len(point["vector"]["dense"]))
print("sparse:", point["vector"]["sparse"])
print("payload keys:", sorted(point["payload"]))


## Schema kỳ vọng của collection

`expected_schema` mô tả named vectors phải tạo khi collection chưa tồn tại: `dense` (size 384, cosine) và `sparse` (SparseVectorParams với index được bật). `ensure_collection` chỉ tạo collection khi absent; nếu collection đã tồn tại, nó phải khớp schema này nếu không pipeline fail closed.

Expected output: dense size 384 distance COSINE; sparse index không phải None.


In [ ]:
from vectorstore.qdrant import expected_schema

schema = expected_schema(settings)
print("vector names:", sorted(schema))
print("dense size:", schema["dense"].size)
print("dense distance:", schema["dense"].distance)
print("sparse index enabled:", schema["sparse"].index is not None)


## Real mode (read-only, opt-in)

Ingestion live (`uv run python -m ingestion.pipeline` từ `backend/`) cần Qdrant local chạy qua Docker Compose và E5 trong local cache — hai việc này cần approval riêng từ người dùng và nằm ngoài default mode của notebook.

Khi collection đã được ingestion tạo (572 points), cell dưới có thể chạy read-only bằng cách set `HUE_RAG_QDRANT_REAL=1`. Cell này chỉ gọi `get_collection`, `count` và `scroll` — không tạo, không upsert, không reset, không delete.

Expected output khi real mode tắt: dòng skip hướng dẫn. Khi real mode bật: collection name, schema dense/sparse, point count 572 và 1 payload mẫu.


In [ ]:
import os

from vectorstore.qdrant import client_from_settings

if os.environ.get("HUE_RAG_QDRANT_REAL") != "1":
    print("Skipped: real mode is off. Set HUE_RAG_QDRANT_REAL=1 only after")
    print("ingestion created the collection (needs Qdrant up and E5 cached).")
else:
    client = client_from_settings(settings)
    name = settings["vector_database"]["collection_name"]
    info = client.get_collection(name)
    params = info.config.params
    print("collection:", name)
    print("dense:", params.vectors["dense"].size, params.vectors["dense"].distance)
    print("sparse index:", params.sparse_vectors["sparse"].index is not None)
    print("point count:", client.count(name, exact=True).count)
    records, _ = client.scroll(name, limit=1, with_payload=True, with_vectors=False)
    print("sample payload:", records[0].payload)


## Kết luận

Phase 4 cung cấp toàn bộ module ingestion offline-safe: client cache, schema validation, deterministic point builder, batch upsert có retry allowlist, count gate và reset command riêng. Bước live (Docker Compose up, E5 offline, tạo collection và upsert 572 points) chỉ chạy sau khi người dùng duyệt hai approval gate riêng.
